In [ ]:
!pip uninstall -y crewai crewai-tools langchain langchain-core langchain-openai langchain-community pydantic langsmith
!pip install crewai==0.30.11 crewai-tools==0.1.6 langchain==0.1.20 langchain-openai==0.1.7 langchain-community==0.0.38 langchain-core==0.1.53 pydantic==2.11.9 langsmith==0.1.50

In [ ]:
!pip install python-dotenv pandas

In [2]:
from google.colab import userdata
import os

# Load OpenAI API Key
openai_api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = openai_api_key
import openai
client = openai.OpenAI(api_key=openai_api_key)

In [3]:
import os
from langchain_openai import ChatOpenAI # Import ChatOpenAI

def get_openai_llm(model="gpt-4", temperature=0.1):
    """
    Returns an instantiated ChatOpenAI LLM object for CrewAI agents.
    """
    return ChatOpenAI(model=model, temperature=temperature)

# Default LLM for quick testing
def get_default_llm():
    """
    Returns an instantiated 'gpt-4' ChatOpenAI LLM object by default.
    """
    return get_openai_llm(model="gpt-4")

In [4]:
import re

def parse_transaction_sms(sms_text):
    """
    Parses a transaction SMS to extract financial details.
    """
    # Define regular expression patterns for targeted extraction
    currency_pattern = r"txn of ([A-Z]{3})"
    site_pattern = r"at (\w+) on"
    bank_pattern = r"on ([\w\s]+ Bank)"
    card_pattern = r"card ending (\d{4})"

    # Extract values using regex search
    currency_match = re.search(currency_pattern, sms_text)
    site_match = re.search(site_pattern, sms_text)
    bank_match = re.search(bank_pattern, sms_text)
    card_match = re.search(card_pattern, sms_text)

    # Return structured dictionary
    return {
        "Currency Code": currency_match.group(1) if currency_match else None,
        "Site": site_match.group(1) if site_match else None,
        "Bank": bank_match.group(1) if bank_match else None,
        "Card Ending": card_match.group(1) if card_match else None
    }

# Example Usage:
sms = "OTP is 001234 for txn of VND 100.00 at AtoZPay on ABCD Bank card ending 6789. Valid till  09:45. Do not share OTP for security reasons ."
extracted_data = parse_transaction_sms(sms)
print(extracted_data)

{'Currency Code': 'VND', 'Site': 'AtoZPay', 'Bank': 'ABCD Bank', 'Card Ending': '6789'}


In [12]:
import json
import re
from typing import List
from crewai import Agent, Crew, Process, Task

# =====================================================================
# 1. DEFINE CREWAI AGENTS
# =====================================================================
investigator = Agent(
    role="Senior Transaction Investigator",
    goal="Compare incoming transaction metadata against historical profile records to identify deviations.",
    backstory=(
        "You are an expert financial forensic analyst. You take a newly extracted transaction "
        "and compare it line-by-line against a user's recent history to spot unexpected anomalies "
        "in card usage, banking channels, or regional currency shifts."
    ),
    verbose=False,
    allow_delegation=False
)

risk_specialist_1 = Agent(
    role="Fraud Risk Assessment Specialist",
    goal="Evaluate behavioral anomalies, cross-border context, and platform safety patterns.",
    backstory=(
        "You evaluate operational risk variables. You determine if a new merchant or "
        "currency shift is common user behavior or indicative of an account takeover/smishing vector."
    ),
    verbose=False,
    allow_delegation=False
)

risk_specialist_2 = Agent(
    role="Fraud Risk Assessment Specialist",
    goal="Compile deviation metrics to calculate a precise numerical fraud score.",
    backstory=(
        "You weigh structural variance mathematically. If fields like the Bank or Card Number "
        "mismatch historical benchmarks, you drastically scale up the final risk vector."
    ),
    verbose=False,
    allow_delegation=False
)

# =====================================================================
# 2. EXTRACTION AND HISTORY UTILITIES
# =====================================================================
def extract_data(sms_text: str) -> dict:
    currency_pattern = r"txn of ([A-Z]{3})"
    site_pattern = r"at (\w+) on"
    bank_pattern = r"on ([\w\s]+ Bank)"
    card_pattern = r"card ending (\d{4})"

    currency_match = re.search(currency_pattern, sms_text)
    site_match = re.search(site_pattern, sms_text)
    bank_match = re.search(bank_pattern, sms_text)
    card_match = re.search(card_pattern, sms_text)

    return {
        "Currency Code": currency_match.group(1) if currency_match else "Unknown",
        "Site": site_match.group(1) if site_match else "Unknown",
        "Bank": bank_match.group(1) if bank_match else "Unknown",
        "Card Ending": card_match.group(1) if card_match else "Unknown"
    }

def fetch_last_ten_transactions() -> List[dict]:
    return [
        {"Currency Code": "VND", "Site": "AtoZPay", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "USD", "Site": "Amazon", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "VND", "Site": "GrabTaxi", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "SGD", "Site": "Netflix", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "VND", "Site": "Shopee", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "VND", "Site": "GongCha", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "USD", "Site": "SteamGames", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "VND", "Site": "Agoda", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "VND", "Site": "CGVCinemas", "Bank": "ABCD Bank", "Card Ending": "6789"},
        {"Currency Code": "EUR", "Site": "Spotify", "Bank": "ABCD Bank", "Card Ending": "6789"}
    ]

# =====================================================================
# 3. PIPELINE WITH ROBUST NATIVE JSON PARSING
# =====================================================================
def analyze_incoming_sms_against_history(incoming_sms: str) -> dict:
    current_txn = extract_data(incoming_sms)
    historical_log = fetch_last_ten_transactions()

    incoming_context = json.dumps(current_txn, indent=2)
    history_context = json.dumps(historical_log, indent=2)

    task_1 = Task(
        description=(
            f"INCOMING TRANSACTION PARAMETERS:\n{incoming_context}\n\n"
            f"HISTORICAL PATTERN MATRIX (LAST 10 TXNS):\n{history_context}\n\n"
            "Analyze the data. Does the Bank name and Card Ending number exist anywhere "
            "within the historical records? Highlight matching vs non-matching fields."
        ),
        expected_output="A structural alignment breakdown separating normal flags from discrepancies.",
        agent=investigator,
        tools=[]
    )

    task_2 = Task(
        description="Review the investigator's matching log. Evaluate behavioral risk. If the incoming merchant site or currency code has never been seen in the historical logs, determine if it constitutes high-risk variance.",
        expected_output="A behavioral threat landscape mapping out context risk patterns.",
        agent=risk_specialist_1,
        tools=[]
    )

    task_3 = Task(
        description=(
            "Synthesize historical matching flags. Assign a definitive fraud score from 0 to 100. "
            "Note: If the Bank or Card Number does not match the historical logs, the score must be very high.\n\n"
            "YOU MUST RESPOND ONLY WITH A VALID JSON OBJECT using this exact schema:\n"
            "{\n"
            '  "fraud_score": <int 0-100>,\n'
            '  "verdict": "<Legitimate | Suspicious | Fraudulent>",\n'
            '  "reasoning": "<text explanation>"\n'
            "}\n"
            "Do not wrap your response in markdown code blocks like ```json."
        ),
        expected_output="A raw valid JSON string containing fraud_score, verdict, and reasoning keys.",
        agent=risk_specialist_2,
        tools=[]
    )

    crew = Crew(
        agents=[investigator, risk_specialist_1, risk_specialist_2],
        tasks=[task_1, task_2, task_3],
        process=Process.sequential
    )

    print("🔎 Extracting SMS values and compiling historical comparison profiles...")

    # Executing synchronously to bypass thread loop issues
    crew_output = crew.kickoff()

    # Handle string extraction depending on CrewAI version output type
    raw_output_str = str(crew_output).strip()

    # Clean up markdown code blocks if the agent accidentally added them
    if raw_output_str.startswith("```"):
        raw_output_str = raw_output_str.strip("`").replace("json", "", 1).strip()

    try:
        parsed_assessment = json.loads(raw_output_str)
    except Exception as e:
        parsed_assessment = {
            "fraud_score": 50,
            "verdict": "Suspicious",
            "reasoning": f"Failed to safely parse agent output string into JSON. Raw string: {raw_output_str}"
        }
        print(f"Parsing failed: {e}")

    return {
        "Extracted Incoming Transaction": current_txn,
        "Risk Assessment Matrix": parsed_assessment
    }

# =====================================================================
# 4. EXECUTION EXAMPLE
# =====================================================================
if __name__ == "__main__":
    valid_sms = "OTP is 001234 for txn of VND 100.00 at AtoZPay on ABCD Bank card ending 6789. Valid till 09:45."
    result_a = analyze_incoming_sms_against_history(valid_sms)
    print("\n📊 RESULT FOR VALID SMS MATCHING PROFILE HISTORY:")
    print(json.dumps(result_a, indent=2))


🔎 Extracting SMS values and compiling historical comparison profiles...

📊 RESULT FOR VALID SMS MATCHING PROFILE HISTORY:
{
  "Extracted Incoming Transaction": {
    "Currency Code": "VND",
    "Site": "AtoZPay",
    "Bank": "ABCD Bank",
    "Card Ending": "6789"
  },
  "Risk Assessment Matrix": {
    "fraud_score": 85,
    "verdict": "Suspicious",
    "reasoning": "The transaction involves a new merchant site and currency code that has never been seen in the historical records. While this doesn't necessarily suggest a high-risk variance, it's accompanied by other suspicious activities and the new merchant site has a low trustworthiness score. Hence, we need to continuously monitor the user's transaction behavior for any sudden changes or anomalies. If these signs persist, it would require further investigation to determine if it's a common user behavior or indicative of an account takeover/smishing vector."
  }
}


In [13]:
from crewai_tools import tool
import json
from datetime import datetime, timedelta

@tool("Classify Customer Query")
def classify_query(query_text: str) -> dict:
    """
    Classify customer query by category and urgency.
    Returns category, urgency level, and suggested routing.
    """
    query_lower = query_text.lower()

    # Category classification
    if any(word in query_lower for word in ["password", "login", "access", "locked"]):
        category = "authentication"
        urgency = "high"
    elif any(word in query_lower for word in ["statement", "balance", "transaction"]):
        category = "account_inquiry"
        urgency = "medium"
    elif any(word in query_lower for word in ["fraud", "unauthorized", "suspicious"]):
        category = "fraud_concern"
        urgency = "critical"
    elif any(word in query_lower for word in ["loan", "credit", "rate", "application"]):
        category = "credit_services"
        urgency = "medium"
    elif any(word in query_lower for word in ["technical", "error", "bug", "not working"]):
        category = "technical_issue"
        urgency = "high"
    else:
        category = "general_inquiry"
        urgency = "low"

    # Routing suggestion
    routing = {
        "authentication": "resolution_agent",
        "account_inquiry": "account_agent",
        "fraud_concern": "escalate_to_fraud_team",
        "credit_services": "credit_specialist",
        "technical_issue": "technical_agent",
        "general_inquiry": "resolution_agent"
    }

    return {
        "category": category,
        "urgency": urgency,
        "suggested_routing": routing.get(category, "manager_review"),
        "keywords_detected": [word for word in ["password", "fraud", "error", "loan"]
                             if word in query_lower]
    }
@tool("Access Customer Account")
def access_customer_account(customer_id: str) -> dict:
    """
    Retrieve customer account information and recent activity.
    Returns account status, balance, recent transactions, and alerts.
    """
    # Simulated account data
    accounts = {
        "CUST12345": {
            "customer_id": "CUST12345",
            "name": "John Smith",
            "account_type": "Premium Checking",
            "status": "active",
            "balance": 15420.50,
            "last_login": "2024-03-24T10:15:00Z",
            "recent_transactions": [
                {"date": "2024-03-25", "description": "ATM Withdrawal", "amount": -200.00},
                {"date": "2024-03-24", "description": "Salary Deposit", "amount": 5500.00},
                {"date": "2024-03-23", "description": "Online Purchase", "amount": -89.99}
            ],
            "alerts": [],
            "customer_since": "2019-06-15",
            "contact_email": "john.smith@email.com",
            "contact_phone": "+1-555-0123"
        },
        "CUST67890": {
            "customer_id": "CUST67890",
            "name": "Sarah Johnson",
            "account_type": "Business Checking",
            "status": "active",
            "balance": 45230.75,
            "last_login": "2024-03-25T14:30:00Z",
            "recent_transactions": [
                {"date": "2024-03-25", "description": "Wire Transfer", "amount": -15000.00},
                {"date": "2024-03-24", "description": "Deposit", "amount": 22000.00}
            ],
            "alerts": ["High-value wire transfer pending verification"],
            "customer_since": "2021-03-10",
            "contact_email": "sarah.j@business.com",
            "contact_phone": "+1-555-0456"
        }
    }

    return accounts.get(customer_id, {"error": "Customer not found"})
@tool("Execute Account Action")
def execute_account_action(customer_id: str, action: str, parameters: dict = None) -> dict:
    """
    Execute common account actions like password reset, statement generation, etc.
    Returns confirmation and next steps.
    """
    if parameters is None:
        parameters = {}

    valid_actions = {
        "reset_password": "Password reset initiated. Temporary password sent to registered email.",
        "request_statement": f"Statement for period {parameters.get('period', 'last 30 days')} generated and sent to email.",
        "update_contact": "Contact information update request submitted. Verification required.",
        "report_fraud": "Fraud report filed. Case number FR-2024-12345 created. Fraud team will contact within 2 hours.",
        "dispute_transaction": "Transaction dispute initiated. Reference number DT-2024-67890. Resolution within 10 business days."
    }

    if action not in valid_actions:
        return {
            "success": False,
            "message": f"Unknown action: {action}",
            "available_actions": list(valid_actions.keys())
        }

    return {
        "success": True,
        "action": action,
        "customer_id": customer_id,
        "message": valid_actions[action],
        "timestamp": datetime.now().isoformat(),
        "reference_number": f"REF-{hash(action + customer_id) % 1000000}"
    }
@tool("Check Knowledge Base")
def check_knowledge_base(query: str) -> dict:
    """
    Search internal knowledge base for solutions and documentation.
    Returns relevant articles and step-by-step solutions.
    """
    # Simulated knowledge base
    knowledge_base = {
        "password reset": {
            "article_id": "KB001",
            "title": "How to Reset Your Password",
            "solution": "1. Click 'Forgot Password' on login page\n2. Enter registered email\n3. Check email for reset link\n4. Create new password (min 12 characters)",
            "related_articles": ["KB002: Account Security", "KB015: Two-Factor Authentication"]
        },
        "transaction dispute": {
            "article_id": "KB045",
            "title": "Disputing a Transaction",
            "solution": "1. Identify the disputed transaction\n2. Gather supporting documentation\n3. Submit dispute form within 60 days\n4. Wait for investigation (5-10 business days)",
            "related_articles": ["KB046: Fraud Protection", "KB047: Chargeback Process"]
        },
        "mobile app error": {
            "article_id": "KB123",
            "title": "Troubleshooting Mobile App Issues",
            "solution": "1. Clear app cache\n2. Update to latest version\n3. Check internet connection\n4. Restart device\n5. Reinstall if problem persists",
            "related_articles": ["KB124: App Features", "KB125: Mobile Security"]
        }
    }

    query_lower = query.lower()
    for key, article in knowledge_base.items():
        if key in query_lower:
            return article

    return {
        "article_id": None,
        "title": "No exact match found",
        "solution": "Please contact customer support for personalized assistance.",
        "related_articles": []
    }

In [14]:
from crewai import Agent
from langchain_openai import ChatOpenAI # Ensure ChatOpenAI is imported for potential direct use if needed

def create_customer_service_agents():
    """
    Create a hierarchical customer service team.
    Uses different LLMs based on task complexity for cost optimization.
    """

    # Intake Agent - Simple classification task, use fast/cheap model
    intake_agent = Agent(
        role="Customer Query Intake Specialist",
        goal="Quickly classify and route customer queries to appropriate handlers",
        backstory="""You are a front-line customer service agent who excels at
        understanding customer needs and routing them efficiently. You identify
        urgency levels and suggest the best specialist to handle each query.""",
        tools=[classify_query],
        llm=get_openai_llm(model="gpt-3.5-turbo-0125"), # Pass instantiated LLM object
        verbose=True,
        allow_delegation=False
    )

    # Account Agent - Data retrieval, use mid-tier model
    account_agent = Agent(
        role="Account Information Specialist",
        goal="Access and explain customer account information accurately",
        backstory="""You are an account specialist with 8 years of experience.
        You retrieve customer data, explain account activity, and identify any
        issues that need attention. You present information clearly and highlight
        anything unusual.""",
        tools=[access_customer_account],
        llm=get_openai_llm(model="gpt-3.5-turbo-0125"), # Pass instantiated LLM object
        verbose=True,
        allow_delegation=False
    )

    # Resolution Agent - Common problem solving
    resolution_agent = Agent(
        role="Problem Resolution Specialist",
        goal="Resolve common customer issues efficiently using standard procedures",
        backstory="""You are a problem-solving expert who handles routine customer
        requests like password resets, statement requests, and contact updates.
        You follow documented procedures, execute actions accurately, and provide
        clear confirmation to customers.""",
        tools=[execute_account_action, check_knowledge_base],
        llm=get_openai_llm(model="gpt-3.5-turbo-0125"), # Pass instantiated LLM object
        verbose=True,
        allow_delegation=False
    )

    # Technical Agent - Complex issues, use stronger model
    technical_agent = Agent(
        role="Technical Support Specialist",
        goal="Diagnose and resolve technical issues with banking systems and applications",
        backstory="""You are a senior technical support engineer with deep knowledge
        of banking systems, mobile applications, and online platforms. You troubleshoot
        complex technical problems, provide detailed solutions, and escalate system-wide
        issues when necessary.""",
        tools=[check_knowledge_base],
        llm=get_openai_llm(model="gpt-4o-mini"), # Pass instantiated LLM object
        verbose=True,
        allow_delegation=False
    )

    # Manager Agent - Coordination and decision-making, use best model
    manager_agent = Agent(
        role="Customer Service Manager",
        goal="Coordinate specialists to resolve customer queries efficiently and ensure quality",
        backstory="""You are an experienced customer service manager who oversees
        a team of specialists. You assess situations, delegate tasks appropriately,
        make escalation decisions, and ensure customers receive excellent service.
        You balance efficiency with quality and know when to escalate to human staff.""",
        tools=[],  # Manager coordinates but doesn't execute actions directly
        llm=get_openai_llm(model="gpt-4"), # Pass instantiated LLM object
        verbose=True,
        allow_delegation=True  # This makes it a manager
    )

    return {
        "intake": intake_agent,
        "account": account_agent,
        "resolution": resolution_agent,
        "technical": technical_agent,
        "manager": manager_agent
    }

In [15]:
from crewai import Task

def create_customer_service_tasks(agents, customer_query, customer_id):
    """
    Create tasks for hierarchical customer service workflow.
    Manager will coordinate these tasks dynamically.
    """

    # Task 1: Classify and Route Query
    intake_task = Task(
        description=f"""Analyze this customer query and determine routing:

        Customer Query: "{customer_query}"
        Customer ID: {customer_id}

        Use the classification tool to:
        1. Identify the query category
        2. Assess urgency level
        3. Recommend routing to appropriate specialist
        4. Extract key information for the handling agent""",

        expected_output="""Query classification containing:
        - Category (authentication, account_inquiry, fraud_concern, etc.)
        - Urgency level (low, medium, high, critical)
        - Recommended routing
        - Key information extracted from query""",

        agent=agents["intake"]
    )

    # Task 2: Retrieve Account Context
    account_task = Task(
        description=f"""Retrieve account information for customer {customer_id}.

        Provide:
        1. Current account status and balance
        2. Recent transaction history
        3. Any active alerts or issues
        4. Relevant account details for addressing the query""",

        expected_output="""Account summary containing:
        - Account status and balance
        - Recent activity
        - Any alerts or concerns
        - Customer profile information""",

        agent=agents["account"],
        context=[intake_task]  # Depends on intake classification
    )

    # Task 3: Resolve or Escalate
    resolution_task = Task(
        description="""Based on the query classification and account context,
        either resolve the customer issue or prepare for escalation.

        If resolvable:
        1. Execute appropriate account action
        2. Provide clear confirmation to customer
        3. Document resolution

        If escalation needed:
        1. Summarize the situation
        2. Identify escalation path
        3. Prepare handoff information""",

        expected_output="""Resolution report containing:
        - Action taken (if resolved)
        - Customer communication (what to tell the customer)
        - Escalation recommendation (if needed)
        - Reference numbers or case IDs""",

        agent=agents["resolution"],
        context=[intake_task, account_task]
    )

    # Task 4: Technical Support (conditional, used if query is technical)
    technical_task = Task(
        description="""Provide technical support for complex system or application issues.

        1. Review the query and account context
        2. Search knowledge base for solutions
        3. Provide step-by-step troubleshooting
        4. Escalate to engineering if system-wide issue""",

        expected_output="""Technical support response containing:
        - Problem diagnosis
        - Step-by-step solution
        - Knowledge base article references
        - Escalation flag if needed""",

        agent=agents["technical"],
        context=[intake_task, account_task]
    )

    return [intake_task, account_task, resolution_task, technical_task]

In [16]:
from crewai import Crew, Process

def handle_customer_query(query: str, customer_id: str):
    """
    Process customer service query using hierarchical workflow.
    """
    print(f"\n{'='*80}")
    print(f"CUSTOMER SERVICE - Query Processing")
    print(f"Customer: {customer_id}")
    print(f"Query: {query}")
    print(f"{'='*80}\n")

    # Create the agent team
    agents = create_customer_service_agents()

    # Create tasks
    tasks = create_customer_service_tasks(agents, query, customer_id)

    # Create hierarchical crew
    # Manager agent coordinates specialists automatically
    crew = Crew(
        agents=[
            agents["intake"],
            agents["account"],
            agents["resolution"],
            agents["technical"]
        ],
        tasks=tasks,
        process=Process.hierarchical,
        manager_llm=agents["manager"].llm,  # Use manager's LLM for coordination
        verbose=True
    )

    # Execute
    try:
        result = crew.kickoff()

        print(f"\n{'='*80}")
        print("FINAL RESPONSE TO CUSTOMER")
        print(f"{'='*80}\n")
        print(result)

        return result

    except Exception as e:
        print(f"\nError processing query: {str(e)}")
        return None


In [ ]:
if __name__ == "__main__":
     #Example 1: Password reset request
     result1 = handle_customer_query(
         query="I can't log into my account. I think I forgot my password.",
         customer_id="CUST12345"
     )

     #Example 2: Account inquiry
    #result2 = handle_customer_query(
    #    query="What's my current balance and recent transactions?",
    #    customer_id="CUST12345"
    #)

    # Example 3: Technical issue
     #result3 = handle_customer_query(
     #    query="The mobile app keeps crashing when I try to view statements.",
     #    customer_id="CUST67890"
     #)